In [22]:
#imports
import spacy
import scispacy
import timeit
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import plotly.express as px


In [2]:
vectorizer = CountVectorizer(lowercase=True,)

In [3]:
#load term parser
nlp = spacy.load("en_core_sci_lg")

In [4]:
#change this to live file ingest later

file_path = Path.cwd().parent / "archs4metadata/OSD-100|Mmus_C57-6J_EYE_FLT_Rep1_M23_archs4_top10hits_metadata.csv"

md_df = pd.read_csv(file_path)

idx = defaultdict(set) #may want to cache later if it gets big

In [5]:
def _add_spacy_terms(): # adds spacy terms to dataframe, returns list of flat terms
    md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

    for row in md_df.itertuples():
        doc = nlp(row.geo_summary)
        for ent in doc.ents:
            term = ent.text.split()
            row.spacey_terms.extend(term)
            for t in term:
                idx[t].add(row.gse)
            
    
    return [item for lst in md_df["spacey_terms"] for item in lst]

In [6]:
def _update_idx():
    
    analyzer = vectorizer.build_analyzer()
    new_idx = defaultdict(set)

    for term,doc_ids in idx.items():
        tokenized = analyzer(term)
        for token in tokenized:
            new_idx[token].update(doc_ids)

    return new_idx  


In [7]:
def _add_journals(cl_df):
     cl_df['journals'] = cl_df.apply(lambda _: [], axis=1)
     cl_df["Representation"] = cl_df["Representation"].apply(lambda lst: [x for x in lst if x])

     for row in cl_df.itertuples():
        
        
        for term in row.Representation:
            
            if term in idx:
               row.journals.extend(list(idx[term]))
            else:
                  print(f"term {term} not found in index")
     
     return cl_df      
         

In [8]:

def cluster_sample(): 
    global idx
    flat_terms = _add_spacy_terms()
    vectorizer.fit(flat_terms)
    idx = _update_idx()
    model = SentenceTransformer('allenai/biomed_roberta_base')
    topic_model = BERTopic(vectorizer_model=vectorizer, embedding_model=model)
    topics,probs = topic_model.fit_transform(flat_terms)
    clusters = _add_journals(topic_model.get_topic_info())   
    return topic_model,clusters
    


In [9]:
def get_cluster_distribution(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
   
    total = len(journals)

    percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
    return percent_dist

In [10]:
def get_cluster_counts(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
    cnts = dict(cnts)
    return cnts

In [11]:
def get_top_cluster_distribution(clusters,n=5):
    top_clusters = clusters.head(n)
    distributions = {}
    for idx, row in top_clusters.iterrows():
        distributions[row.Topic] = get_cluster_distribution(clusters,idx)
    return distributions

In [12]:
def get_top_journals(clusters,n=5):
    
    top_clusters = clusters.head(n).iloc[1:] #skip outliers
    overall_cnts = defaultdict(int)
    for idx, row in top_clusters.iterrows():
       # print(f"Processing cluster {row.Topic} at index {idx}")
        journal_counts = get_cluster_counts(clusters,idx)
       # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
        for journal, count in journal_counts.items():
            #print(f"Adding {count} to overall count for journal {journal}")
            overall_cnts[journal] += count
    
    return overall_cnts

In [13]:
def get_top_journal_distribution(clusters):
    top_cnt_dict = get_top_journals(clusters)
    total = sum(top_cnt_dict.values())
    percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}
    return percent_dist


In [ ]:
model,clusters = cluster_sample() #47s no gpu on VM 

No sentence-transformers model found with name allenai/biomed_roberta_base. Creating a new one with mean pooling.


In [28]:
jour_dist

{'GSE205070': 47.72727272727273,
 'GSE210492': 22.727272727272727,
 'GSE124745': 11.363636363636363,
 'GSE88819': 4.545454545454546,
 'GSE143281': 13.636363636363635}

In [41]:
top_j = get_top_journals(clusters)
jour_dict = get_top_journal_distribution(clusters)
jour_dist = pd.DataFrame(list(get_top_journal_distribution(clusters).items()),columns=['Journal','Percentage'])

In [32]:
jour_dist

,Journal,Percentage
0,GSE205070,47.727273
1,GSE210492,22.727273
2,GSE124745,11.363636
3,GSE88819,4.545455
4,GSE143281,13.636364


In [ ]:
get_top_cluster_distribution(clusters) #note: includes outliers as -1 index

In [47]:
list(jour_dict.keys())

['GSE205070', 'GSE210492', 'GSE124745', 'GSE88819', 'GSE143281']

In [51]:
list(jour_dict.items())

[('GSE205070', 47.72727272727273),
 ('GSE210492', 22.727272727272727),
 ('GSE124745', 11.363636363636363),
 ('GSE88819', 4.545454545454546),
 ('GSE143281', 13.636363636363635)]

In [52]:
def _plot_dist(dist):
    fig = px.pie(
    names = list(jour_dict.keys()),
    values = list(jour_dict.values()),
   
    title="Percentage Breakdown of Journals",
    )

    fig.show()


In [53]:
_plot_dist(jour_dist)